# Projet NLP - Sentiment des tweets aeriens

Objectif: predire le sentiment (`negative`, `neutral`, `positive`) de tweets adresses a six compagnies aeriennes, puis fournir une demonstration deployable.

## 0. Cadrage business

Problematique: classer automatiquement les tweets pour prioriser les reponses du service client, suivre la satisfaction par compagnie et detecter les signaux de crise.

Questions business:

1. Quelle compagnie a la plus forte part de tweets negatifs ?
2. Quels motifs expliquent les tweets negatifs ?
3. Quel modele offre le meilleur compromis F1-macro / cout de calcul / deployabilite ?

Metrique principale: F1-macro, car la classe negative domine le dataset.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

from src.project_pipeline import load_dataset, create_splits, clean_tweet

df = load_dataset()
display(df.head())
display(df['airline_sentiment'].value_counts(normalize=True).rename('part'))

## 1. Exploration et preparation

Le pipeline nettoie les tweets et cree un split stratifie fige.

In [ ]:
splits = create_splits(df)
print(len(splits.train), len(splits.validation), len(splits.test))
display(splits.train[['airline_sentiment', 'text', 'clean_text']].head())

Les figures d'EDA sont generees par `python -m src.project_pipeline`.

In [ ]:
for path in sorted(Path('outputs/figures').glob('0*.png')):
    print(path)
    display(Image(filename=str(path)))

## 2. Representations

- TF-IDF: baseline explicable avec n-grammes.
- Word2Vec-style: skip-gram entraine sur le corpus, reutilise comme initialisation des CNN/LSTM/Attention.

Les voisins semantiques sont sauvegardes dans `outputs/tables/word2vec_voisins.csv`.

In [ ]:
neighbors = Path('outputs/tables/word2vec_voisins.csv')
if neighbors.exists():
    display(pd.read_csv(neighbors).head(20))
else:
    print('Lancez python -m src.project_pipeline pour generer ce tableau.')

## 3. Modelisation

Modeles entraines: regression logistique TF-IDF, ANN TF-IDF, CNN Word2Vec, BiLSTM Word2Vec, BiLSTM avec attention.

In [ ]:
results_path = Path('outputs/tables/comparaison_modeles.csv')
if results_path.exists():
    results = pd.read_csv(results_path)
    display(results[['model', 'accuracy', 'macro_f1', 'weighted_f1']])
else:
    print('Lancez python -m src.project_pipeline pour entrainer les modeles.')

## 4. Evaluation et analyse d'erreurs

In [ ]:
for path in sorted(Path('outputs/figures').glob('confusion_*.png')):
    print(path.name)
    display(Image(filename=str(path)))

errors = Path('outputs/tables/analyse_erreurs_neutre.csv')
if errors.exists():
    display(pd.read_csv(errors).head(10))

## 5. Interpretabilite

In [ ]:
for path in ['outputs/figures/interpretabilite_top_tfidf_terms.png', 'outputs/figures/interpretabilite_attention_exemple.png']:
    p = Path(path)
    if p.exists():
        print(p.name)
        display(Image(filename=str(p)))

## 6. Deploiement

L'application se lance depuis un terminal avec:

```bash
streamlit run app/streamlit_app.py
```